# 03 — SVM with a Precomputed Kernel

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | `**03**` SVM with kernel | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | [`06` Weighted (WGKM) kernel](06_weighted_kernel.ipynb) | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | [`09` Spectrum encoder & NB](09_spectrum_encoder_and_differential.ipynb) | [`10` Gappy encoder](10_gappy_encoder.ipynb) | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | [`12` Shuffler & chunker](12_shuffler_and_chunker.ipynb)

Train an SVM on a gkm kernel using the `KernelSVM` model. Compares three weight schemes (`full`, `estimated_full`, `truncated`) and the effect of `use_rc`.

In [1]:
import numpy as np
from kmer.kernels import GKMKernel
from kmer.models import KernelSVM
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)
def random_seq(n, rng):
    return "".join(rng.choice(list("ACGT"), size=n))

rng = np.random.default_rng(42)
positives = [("ACGTACGT" + random_seq(12, rng)) for _ in range(60)]
negatives = [random_seq(20, rng) for _ in range(60)]
seqs = positives + negatives
y = np.array([1]*60 + [0]*60)

seqs_train, seqs_test, y_train, y_test = train_test_split(
    seqs, y, test_size=0.3, stratify=y, random_state=42
)
print(f"Train: {len(seqs_train)}, Test: {len(seqs_test)}")

Train: 84, Test: 36


## Compare three weight schemes

In [2]:
schemes = ["full", "estimated_full", "truncated"]
results = {}
for scheme in schemes:
    kern = GKMKernel(L=10, k=6, d=3, kernel_type=scheme, use_rc=True)
    clf = KernelSVM(kern, C=1.0)
    clf.fit(seqs_train, y_train)
    preds = clf.predict(seqs_test)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, clf.decision_function(seqs_test))
    results[scheme] = {"kern": kern, "clf": clf, "acc": acc, "auc": auc}
    print(f"{scheme:18s}: acc={acc:.3f}, AUC={auc:.3f}")

full              : acc=0.972, AUC=0.944
estimated_full    : acc=0.972, AUC=0.944


truncated         : acc=0.972, AUC=0.944


## The effect of `use_rc`

In [3]:
# Build a dataset where half the positives have the RC motif
positives_rc = [("ACGTACGT"[::-1].translate(str.maketrans("ACGT", "TGCA")) + random_seq(12, rng))
                for _ in range(30)]
positives_fwd = [("ACGTACGT" + random_seq(12, rng)) for _ in range(30)]
positives_mix = positives_fwd + positives_rc
negatives = [random_seq(20, rng) for _ in range(60)]
seqs_mix = positives_mix + negatives
y_mix = np.array([1]*60 + [0]*60)
seqs_tr, seqs_te, y_tr, y_te = train_test_split(seqs_mix, y_mix, test_size=0.3, stratify=y_mix, random_state=42)

for use_rc in [True, False]:
    kern = GKMKernel(L=10, k=6, d=3, kernel_type="truncated", use_rc=use_rc)
    clf = KernelSVM(kern, C=1.0)
    clf.fit(seqs_tr, y_tr)
    auc = roc_auc_score(y_te, clf.decision_function(seqs_te))
    print(f"use_rc={use_rc}: AUC={auc:.3f}")

use_rc=True: AUC=1.000
use_rc=False: AUC=1.000
